In [ ]:
# @title  CIVITAI FILE NAME CHECKER
import requests

# Example Model Version ID from the URL (1450053)
model_version_id = "1065263"
api_url = f"https://civitai.com/api/v1/model-versions/{model_version_id}"

data = requests.get(api_url).json()

# Grab the primary file's name
for file in data.get('files', []):
    print(f"Official Filename: {file['name']}")

Official Filename: LapLyingBlowjob-v2_Illustrious.safetensors


In [ ]:
# @title 🗑️ COMFYUI BATCH DELETER
import os
import shutil

# ============================================================
# ⚙️ SELECT WHAT TO DELETE
# Set any option to True to delete its contents/folder.
# ============================================================

# 1. Clean individual file types (Keeps the folder, removes files)
CLEAR_LORAS = False
CLEAR_CHECKPOINTS = False
CLEAR_CONTROLNETS = False
CLEAR_VAES = False
CLEAR_CLIPS = False
CLEAR_UPSCALE_MODELS = False

# 2. Clean generated outputs (Outputs, Temp caches, Logs)
CLEAR_OUTPUT_IMAGES = False
CLEAR_TEMP_CACHE = False

# 3. Nuclear options (Deletes entire directories)
REMOVE_ALL_CUSTOM_NODES = False
WIPE_ENTIRE_COMFYUI = True  # ⚠️ Wipes everything including ComfyUI installation

# 4. Delete specific files by name (e.g., ["corrupt_model.safetensors", "bad_lora.safetensors"])
SPECIFIC_FILES_TO_DELETE = [
    # "example_file_name.safetensors",
]


# ============================================================
# 🚀 DELETER ENGINE
# ============================================================
BASE_DIR = "/content/ComfyUI"
MODELS_DIR = os.path.join(BASE_DIR, "models")

print("="*60)
print("🗑️ COMFYUI BATCH DELETER INITIALIZED")
print("="*60)

def clear_folder_contents(folder_path, label):
    """Deletes all files inside a directory without removing the directory itself."""
    if not os.path.exists(folder_path):
        print(f"⏩ {label} directory does not exist. Skipping.")
        return

    count = 0
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
                count += 1
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
                count += 1
        except Exception as e:
            print(f"❌ Failed to delete {filename}: {e}")

    print(f"✅ Cleared {label}: {count} items removed.")

# --- Execute Folder Cleanups ---
if CLEAR_LORAS:
    clear_folder_contents(os.path.join(MODELS_DIR, "loras"), "LoRAs")

if CLEAR_CHECKPOINTS:
    clear_folder_contents(os.path.join(MODELS_DIR, "checkpoints"), "Checkpoints")

if CLEAR_CONTROLNETS:
    clear_folder_contents(os.path.join(MODELS_DIR, "controlnet"), "ControlNets")

if CLEAR_VAES:
    clear_folder_contents(os.path.join(MODELS_DIR, "vae"), "VAEs")

if CLEAR_CLIPS:
    clear_folder_contents(os.path.join(MODELS_DIR, "clip"), "CLIPs")

if CLEAR_UPSCALE_MODELS:
    clear_folder_contents(os.path.join(MODELS_DIR, "upscale_models"), "Upscalers")

if CLEAR_OUTPUT_IMAGES:
    clear_folder_contents(os.path.join(BASE_DIR, "output"), "Outputs")

if CLEAR_TEMP_CACHE:
    clear_folder_contents(os.path.join(BASE_DIR, "temp"), "Temp Cache")

if REMOVE_ALL_CUSTOM_NODES:
    clear_folder_contents(os.path.join(BASE_DIR, "custom_nodes"), "Custom Nodes")

# --- Execute Specific File Deletions ---
if SPECIFIC_FILES_TO_DELETE:
    print("\n🔍 Searching for specific target files...")
    deleted_specifics = 0
    for root, _, files in os.walk(BASE_DIR):
        for file in files:
            if file in SPECIFIC_FILES_TO_DELETE:
                full_path = os.path.join(root, file)
                os.remove(full_path)
                print(f"✅ Deleted specific file: {file}")
                deleted_specifics += 1
    if deleted_specifics == 0:
        print("⏩ No matching specific files found.")

# --- Execute Nuclear Wipe ---
if WIPE_ENTIRE_COMFYUI:
    if os.path.exists(BASE_DIR):
        print("\n⚠️ Wiping entire ComfyUI directory...")
        shutil.rmtree(BASE_DIR)
        print("💥 ComfyUI has been completely deleted.")

print("\n" + "="*60)
print("🎉 DELETER OPERATION COMPLETE")
print("="*60)

In [2]:
# @title ComfyUI Asset Downloader (Hybrid Naming)
# ============================================================
# 📁 Cell 1: ComfyUI Asset Downloader (Hybrid Naming)
# ============================================================
import os
import subprocess
import re
import getpass
import shutil
from pathlib import Path
from urllib.parse import urlparse, unquote
import requests
from tqdm import tqdm
from IPython.display import clear_output

# ------------------------------------------------------------
# ⚙️ SYSTEM & DIRECTORIES
# ------------------------------------------------------------
clear_output(wait=True)
print("="*60)
print("📥 COMFYUI ASSET DOWNLOADER (HYBRID NAMING)")
print("="*60)

BASE_DIR = Path("/content/ComfyUI")
MODELS_DIR = BASE_DIR / "models"
NODES_DIR = BASE_DIR / "custom_nodes"

for d in [MODELS_DIR / "checkpoints", MODELS_DIR / "vae",
          MODELS_DIR / "controlnet", MODELS_DIR / "loras",
          MODELS_DIR / "clip", MODELS_DIR / "upscale_models",
          MODELS_DIR / "inpaint", NODES_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("✅ Directories created.")

# ------------------------------------------------------------
# 🔑 API & AUTHENTICATION
# ------------------------------------------------------------
print("\n" + "="*60)
print("🔑 API & AUTHENTICATION SETUP")
print("="*60)
print("ℹ️ Press Enter to skip if downloading public models only.\n")

civitai_key = getpass.getpass("Enter Civitai API Key (optional): ").strip() or None
hf_token = getpass.getpass("Enter HuggingFace Token (optional): ").strip() or None

# ------------------------------------------------------------
# 🔧 INSTALL DOWNLOAD ENGINES
# ------------------------------------------------------------
print("\n📦 Checking download engines (aria2 & tqdm)...")
subprocess.run("apt-get update -qq && apt-get install aria2 -y -qq", shell=True)
subprocess.run("pip install tqdm requests -q", shell=True)
print("✅ Downloader engines ready.")

# ------------------------------------------------------------
# 🛠️ HELPER FUNCTIONS
# ------------------------------------------------------------
def resolve_civitai_filename(url):
    headers = {}
    if civitai_key:
        headers["Authorization"] = f"Bearer {civitai_key}"
    try:
        match = re.search(r'/models/(\d+)', url)
        if match and "civitai.com/api/download/models/" in url:
            model_version_id = match.group(1)
            api_url = f"https://civitai.com/api/v1/model-versions/{model_version_id}"
            resp = requests.get(api_url, headers=headers, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                files = data.get("files", [])
                if files:
                    for f in files:
                        if f.get("primary", False):
                            return f.get("name")
                    return files[0].get("name")
    except Exception:
        pass
    return None

def get_clean_url_and_filename(raw_url):
    if "filename=" in raw_url:
        match = re.search(r'filename=([^&]+)', raw_url)
        if match:
            clean_url = raw_url.split('?')[0]
            return clean_url, unquote(match.group(1))
    if "civitai.com/api/download/models/" in raw_url:
        api_filename = resolve_civitai_filename(raw_url)
        if api_filename:
            return raw_url, api_filename
    parsed = urlparse(raw_url)
    clean_url = f"{parsed.scheme}://{parsed.netloc}{parsed.path}"
    filename = unquote(os.path.basename(clean_url))
    if filename.isdigit():
        filename += ".safetensors"
    if not filename:
        filename = "model.safetensors"
    return clean_url, filename

def is_file_valid(filepath, min_size_mb=1):
    if not os.path.exists(filepath):
        return False
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    if size_mb < min_size_mb:
        return False
    try:
        with open(filepath, 'rb') as f:
            header = f.read(100)
            if header.startswith(b'<html') or header.startswith(b'<!DOCTYPE'):
                return False
    except Exception:
        return False
    return True

def download_file(url, output_dir, filename):
    output_path = os.path.join(output_dir, filename)
    if is_file_valid(output_path):
        size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"⏩ Skipping {filename} (Already exists: {size_mb:.1f} MB)")
        return True, "skipped"
    if os.path.exists(output_path):
        os.remove(output_path)
    print(f"\n⬇️ Downloading: {filename}")

    download_url = url
    auth_headers = {}
    if "civitai.com" in url and civitai_key:
        delimiter = "&" if "?" in download_url else "?"
        download_url = f"{download_url}{delimiter}token={civitai_key}"
    elif "huggingface.co" in url and hf_token:
        auth_headers["Authorization"] = f"Bearer {hf_token}"

    cmd = [
        "aria2c",
        "--console-log-level=error",
        "-c", "-x", "16", "-s", "16",
        "-k", "1M", "--timeout=60",
        "--max-tries=3",
        "-d", output_dir,
        "-o", filename,
        download_url
    ]
    for k, v in auth_headers.items():
        cmd.extend(["--header", f"{k}: {v}"])

    subprocess.run(cmd)

    if is_file_valid(output_path):
        size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"✅ Downloaded successfully ({size_mb:.1f} MB)")
        return True, "downloaded"

    print("⚠️ aria2 failed or output invalid. Trying HTTP fallback...")
    try:
        resp = requests.get(download_url, headers=auth_headers, stream=True,
                            timeout=60, allow_redirects=True)
        if resp.status_code == 200:
            total_size = int(resp.headers.get('content-length', 0))
            with open(output_path, "wb") as f:
                with tqdm(total=total_size, unit='B', unit_scale=True,
                          desc=filename[:25]) as pbar:
                    for chunk in resp.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))
            if is_file_valid(output_path):
                size_mb = os.path.getsize(output_path) / (1024 * 1024)
                print(f"✅ Downloaded via fallback ({size_mb:.1f} MB)")
                return True, "downloaded"
    except Exception as e:
        print(f"❌ Fallback failed: {e}")

    if os.path.exists(output_path):
        os.remove(output_path)
    print(f"❌ Failed to download {filename}")
    return False, "failed"

def git_clone(repo_url, target_dir):
    """Clone a Git repository into target_dir with proper CWD."""
    target_dir = Path(target_dir)
    if target_dir.exists():
        print(f"⏩ Custom node '{target_dir.name}' already installed. Skipping.")
        return True, "skipped"

    print(f"📦 Cloning custom node: {target_dir.name}...")
    # Ensure parent directory exists (just in case)
    target_dir.parent.mkdir(parents=True, exist_ok=True)

    # Use BASE_DIR as the working directory to avoid 'Unable to read current working directory'
    cmd = ["git", "clone", "--depth", "1", repo_url, str(target_dir)]
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(BASE_DIR))

    if result.returncode == 0:
        print("✅ Custom node installed.")
        return True, "downloaded"
    else:
        print(f"❌ Failed to clone {target_dir.name}")
        print(f"Error: {result.stderr.strip()}")
        # Clean up partial clone
        if target_dir.exists():
            shutil.rmtree(target_dir, ignore_errors=True)
        return False, "failed"

# ------------------------------------------------------------
# 📋 DOWNLOAD LISTS (Hybrid: strings or tuples)
# ------------------------------------------------------------

CHECKPOINTS = [

    # MeinaMix (Anime/Illustration model) - ~2GB
    #"https://civitai.com/api/download/models/119057",

    #MeinaHentai
    #"https://civitai.com/api/download/models/948699?fileId=855570",

    # SDPose Wholebody - ~1.9GB
    #"https://huggingface.co/Comfy-Org/SDPose/resolve/main/checkpoints/sdpose_wholebody_fp16.safetensors?download=true",

    # Illustrious SDXL (Large model - ~6.5GB)
    "https://civitai.com/api/download/models/2883731",

    #PornWorks Anime Desire ● NSFW Anime & Hentai SDXL/Pony Chekpoint /// pornworksAnimeDesireNSFW_illustrious.safetensors
    "https://civitai.com/api/download/models/2275504?fileId=2167430",

    #Nova Anime XL /// novaAnimeXL_ilV190.safetensors
    #"https://civitai.com/api/download/models/2940478?fileId=2819621",

    #Ultimate Hentai Anime RX - ( T-Rex ) - Anime ScreenShot - | CHECKPOINT - Illustrious XL | - by YeiyeiArt /// ultimateHentaiAnimeRXTRexAnime_rxV1.safetensors
    "https://civitai.com/api/download/models/1828803?fileId=1729137",
]

VAES = [

    # SD 1.5 VAE - ~300MB
    #"https://civitai.com/api/download/models/311162?fileId=262135",

    # SDXL VAE - ~300MB
    "https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors",

]

CONTROLNETS = [

    # OpenPose SD15
    #"https://civitai.com/api/download/models/537364?fileId=453956",

    # OpenPose SDXL
    #"https://huggingface.co/dimitribarbot/controlnet-openpose-sdxl-1.0-safetensors/resolve/main/diffusion_pytorch_model.safetensors?download=true",

    # Canny SD15
    #"https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_canny.pth",

    # Depth SD15
    #"https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11f1p_sd15_depth.pth",

    # Xinsir ControlNet Union ProMax (All-in-one Pose, Canny, Depth for SDXL/Illustrious)
    "https://huggingface.co/xinsir/controlnet-union-sdxl-1.0/resolve/main/diffusion_pytorch_model_promax.safetensors?download=true",

]

LORAS = [

    #///ILLUSTRIOUS///

    #Brazilian Miku LoRa | Illustrious /// brazilianmiku.safetensors
    "https://civitai.com/api/download/models/1450053?fileId=1351324",
    #[IllustriousXL v0.1] Brazilian Miku | Vocaloid /// vocaloid_brazilianmiku_illustriousXL.safetensors
    "https://civitai.com/api/download/models/1109601?fileId=1014519",

    #Illustrious Style Pack-IFL /// IFL_v1.0_IL.safetensors
    "https://civitai.com/api/download/models/2211883?fileId=2104890",
    #Illustrious Style Pack-LMB v2 /// LMB_style_v2.2_IL.safetensors
    "https://civitai.com/api/download/models/2123977?fileId=2018066",
    #Illustrious Style Pack-MSS v2 /// MSS_v2_IL.safetensors
    "https://civitai.com/api/download/models/1942096?fileId=1839728",
    #Illustrious Style Pack-PHM v3 /// PHM_style_IL_v3.3.safetensors
    "https://civitai.com/api/download/models/1570070?fileId=1470025",
    #T-Rex Studio V2 NEW!!- Hentai +18 - | STYLE | PONY XL | Illustrious XL | - COMMISSION - by YeiyeiArt /// ATRex_style-12V2Rev.safetensors
    "https://civitai.com/api/download/models/1804885?fileId=1705538",

    #POV Seated Blowjob /// DPSPBHHC4BBT7THEKB54T3FS30.safetensors
    "https://civitai.com/api/download/models/2817417?fileId=2703493",
    #W Legs Top Down Bottom Up pose /// W_Legs_Top_Down_Bottom_Up_pose.safetensors
    "https://civitai.com/api/download/models/2670299?fileId=2557960",

    #sexy details
    "https://civitai.com/api/download/models/3002225?fileId=2881442",

    #Pillow Bite IL&NoobAI | Shrekman Hentai Loras
    "https://civitai.com/api/download/models/2755658?fileId=2642050",

    #Missionary Variations Pack IL | Shrekman Hentai Loras(Hand holding)
    "https://civitai.com/api/download/models/1418529?fileId=1321058",

    #Missionary Variations Pack IL | Shrekman Hentai Loras(Arm pull)
    "https://civitai.com/api/download/models/2567354?fileId=2455633",

    #Paizuri Variations Pack [Solo] | Shrekman Hentai Loras(Straddling Paizuri)
    "https://civitai.com/api/download/models/2516152?fileId=2404028",

    #Paizuri Variations Pack [Solo] | Shrekman Hentai Loras(Perpendicular Paizuri)
    "https://civitai.com/api/download/models/1550370?fileId=1450627",

    #Orgasm/Climax Enhancers Pack | Shrekman Hentai Lora(Eye Roll)
    "https://civitai.com/api/download/models/2589180?fileId=2476542",

    #Pov Reverse Cowgirl Position LoRa | PonyXL & Illustrious
    "https://civitai.com/api/download/models/1412807?fileId=1314658",

    #Sitting on Lap | PonyXL & Illustrious
    "https://civitai.com/api/download/models/1307377?fileId=1211542",

    #Prone Bone Position LoRa | PonyXL & Illustrious
    "https://civitai.com/api/download/models/1401062?fileId=1303324",

    #Buttjob LoRa | PonyXL & Illustrious
    "https://civitai.com/api/download/models/1282049?fileId=1186552",

    #Lotus Position LoRa | PonyXL & Illustrious
    "https://civitai.com/api/download/models/1409776?fileId=1311642",

    #Mating press position LoRa | Illustrious
    "https://civitai.com/api/download/models/1833284?fileId=1733491",

    #POV Amazon Position
    "https://civitai.com/api/download/models/1488233?fileId=1388546",

    #Amazon position LoRa | Illustrious
    "https://civitai.com/api/download/models/1833287?fileId=1733496",

    #Lying on Lap Blowjob PONY/Illustrious
    "https://civitai.com/api/download/models/1065263?fileId=970759",

    #POV Thighjob on Top / Lying on Top Thighjob - PONY/ILLUSTRIOUS
    "https://civitai.com/api/download/models/1086822?fileId=991947",

    #POV Spooning from Behind / Side Sex from Back
    "https://civitai.com/api/download/models/1055765?fileId=965698",

    #Lying on lap fellatio LoRa | Illustrious
    "https://civitai.com/api/download/models/1834407?fileId=1734586",


]          # add strings or tuples

CLIPS = [

    # Standalone CLIP text encoders (if required by custom workflows)

    # OpenAI CLIP Large
    #"https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/model.safetensors",
]

CLIP_VISION = [
    # ViT-H Image Encoder (Required by IP-Adapter Plus to read reference images)
    # Using a (URL, filename) tuple ensures the file isn't saved as generic "model.safetensors"
    (
        "https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors",
        "CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors"
    ),
]

UPSCALE_MODELS = [

    "https://huggingface.co/uwg/upscaler/resolve/main/ESRGAN/4x_NMKD-Superscale-SP_178000_G.pth",
    "https://huggingface.co/Acly/Omni-SR/resolve/b28a7caa0baba8669de3e49687e76e12448ea020/OmniSR_X2_DIV2K.safetensors?download=true",
    "https://huggingface.co/FacehugmanIII/4x_foolhardy_Remacri/resolve/main/4x_foolhardy_Remacri.pth",
]

INPAINT_MODELS = [

    "https://huggingface.co/Acly/MAT/resolve/main/MAT_Places512_G_fp16.safetensors",

]

# ---- Custom Nodes (Git repos) ----
CUSTOM_NODES = [
    # Core Engine & Management
    "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git",
    "https://github.com/WASasquatch/was-node-suite-comfyui.git",
    "https://github.com/cubiq/ComfyUI_essentials.git",

    # ControlNet & Preprocessors
    "https://github.com/Kosinkadink/ComfyUI-Advanced-ControlNet.git",
    "https://github.com/Fannovel16/comfyui_controlnet_aux.git", # DWpose, OpenPose, Lineart extractors

    # IP-Adapter & Character Reference
    "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git",

    # Segmentation & Masking (Updated)
    "https://github.com/kijai/ComfyUI-segment-anything-2.git",
    "https://github.com/Acly/comfyui-inpaint-nodes.git",
    "https://github.com/Acly/comfyui-tooling-nodes.git",

    # Canvas & Interactive Masking Editors
    "https://github.com/Zlata-Salyukova/Comfy-Canvas.git",
    "https://github.com/Azornes/Comfyui-LayerForge.git",
    "https://github.com/o-l-l-i/ComfyUI-Olm-SplineMask.git",

    # Impact & Inspire Suites (Face Detailers, Regional Samplers)
    "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
    "https://github.com/ltdrdata/ComfyUI-Inspire-Pack.git",

    # Animation & Video Pipelines
    "https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",

    # UI, Layering & Compositing
    "https://github.com/chflame163/ComfyUI_LayerStyle.git",
    "https://github.com/willmiao/ComfyUI-Lora-Manager.git",
    "https://github.com/BlenderNeko/ComfyUI_Cutoff.git",

    # Attention-based regional masking (No color bleed)
    "https://github.com/ramyma/A8R8_ComfyUI_nodes.git",

    # Anima / Transformer Regional Conditioning
    "https://github.com/Sen-sou/Comfyui-Anima-Regional-Conditioning.git",

    # ComfyUI-Prompt-Control
    "https://github.com/asagi4/comfyui-prompt-control.git"
]

EMBEDDINGS = [
    # Common quality or negative embedding files if needed
]

IPADAPTER_MODELS = [
    # SDXL IP-Adapter Plus (For pattern, outfit, and general style reference transfer)
    "https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors",

    # SDXL IP-Adapter Plus Face (For transferring character facial features)
    "https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus-face_sdxl_vit-h.safetensors",
]

# ------------------------------------------------------------
# 🚀 DOWNLOAD ENGINE
# ------------------------------------------------------------
print("\n" + "="*60)
print("🚀 PROCESSING DOWNLOAD QUEUE")
print("="*60)

DOWNLOAD_MAP = {
    NODES_DIR: CUSTOM_NODES,
    MODELS_DIR / "checkpoints": CHECKPOINTS,
    MODELS_DIR / "vae": VAES,
    MODELS_DIR / "controlnet": CONTROLNETS,
    MODELS_DIR / "loras": LORAS,
    MODELS_DIR / "clip": CLIPS,                        # Text Encoders (CLIP_L, T5XXL)
    MODELS_DIR / "clip_vision": CLIP_VISION,          # Image Encoders (ViT-H for IP-Adapter)
    MODELS_DIR / "ipadapter": IPADAPTER_MODELS,        # IP-Adapter Models
    MODELS_DIR / "upscale_models": UPSCALE_MODELS,
    MODELS_DIR / "inpaint": INPAINT_MODELS,
    MODELS_DIR / "embeddings": EMBEDDINGS,            # Textual Inversion / Bad Hands embeddings
}

stats = {"downloaded": 0, "skipped": 0, "failed": 0}

for target_dir, items in DOWNLOAD_MAP.items():
    if not items:
        continue
    category_name = target_dir.name.upper()
    print(f"\n📂 CATEGORY: {category_name}")
    print("-" * 40)

    for entry in items:
        # Skip comments
        if isinstance(entry, str) and entry.strip().startswith("#"):
            continue

        # Custom nodes
        if target_dir == NODES_DIR:
            repo_url = entry
            repo_name = Path(repo_url).stem  # removes .git suffix
            success, status = git_clone(repo_url, target_dir / repo_name)
            stats[status] += 1
            continue

        # Model files
        if isinstance(entry, str):
            url = entry
            clean_url, filename = get_clean_url_and_filename(url)
        elif isinstance(entry, tuple) and len(entry) == 2:
            url, filename = entry
        else:
            print(f"⚠️ Skipping invalid entry: {entry}")
            stats["failed"] += 1
            continue

        success, status = download_file(url, str(target_dir), filename)
        stats[status] += 1

# ------------------------------------------------------------
print("\n" + "="*60)
print(f"🎉 BATCH COMPLETE: {stats['downloaded']} downloaded | "
      f"{stats['skipped']} skipped | {stats['failed']} failed")
print("="*60)

📥 COMFYUI ASSET DOWNLOADER (HYBRID NAMING)
✅ Directories created.

🔑 API & AUTHENTICATION SETUP
ℹ️ Press Enter to skip if downloading public models only.

Enter Civitai API Key (optional): ··········
Enter HuggingFace Token (optional): ··········

📦 Checking download engines (aria2 & tqdm)...
✅ Downloader engines ready.

🚀 PROCESSING DOWNLOAD QUEUE

📂 CATEGORY: CUSTOM_NODES
----------------------------------------
📦 Cloning custom node: ComfyUI-Manager...
✅ Custom node installed.
📦 Cloning custom node: ComfyUI-Custom-Scripts...
✅ Custom node installed.
📦 Cloning custom node: was-node-suite-comfyui...
✅ Custom node installed.
📦 Cloning custom node: ComfyUI_essentials...
✅ Custom node installed.
📦 Cloning custom node: ComfyUI-Advanced-ControlNet...
✅ Custom node installed.
📦 Cloning custom node: comfyui_controlnet_aux...
✅ Custom node installed.
📦 Cloning custom node: ComfyUI_IPAdapter_plus...
✅ Custom node installed.
📦 Cloning custom node: ComfyUI-segment-anything-2...
✅ Custom node ins

brazilianmiku.safetensors: 100%|██████████| 228M/228M [00:06<00:00, 34.5MB/s]


✅ Downloaded via fallback (217.9 MB)

⬇️ Downloading: vocaloid_brazilianmiku_illustriousXL.safetensors
⚠️ aria2 failed or output invalid. Trying HTTP fallback...


vocaloid_brazilianmiku_il: 100%|██████████| 57.5M/57.5M [00:01<00:00, 40.6MB/s]


✅ Downloaded via fallback (54.8 MB)

⬇️ Downloading: IFL_v1.0_IL.safetensors
⚠️ aria2 failed or output invalid. Trying HTTP fallback...


...

✅ Downloaded via fallback (109.1 MB)

⬇️ Downloading: Spooningv2-Illustrious_Noob.safetensors
⚠️ aria2 failed or output invalid. Trying HTTP fallback...


Spooningv2-Illustrious_No: 100%|██████████| 228M/228M [00:03<00:00, 66.8MB/s]


✅ Downloaded via fallback (217.9 MB)

⬇️ Downloading: lying_on_lap_fellatio.safetensors
⚠️ aria2 failed or output invalid. Trying HTTP fallback...


lying_on_lap_fellatio.saf: 100%|██████████| 228M/228M [00:04<00:00, 46.2MB/s]


✅ Downloaded via fallback (217.9 MB)

📂 CATEGORY: CLIP_VISION
----------------------------------------

⬇️ Downloading: CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors
✅ Downloaded successfully (2411.2 MB)

📂 CATEGORY: IPADAPTER
----------------------------------------

⬇️ Downloading: ip-adapter-plus_sdxl_vit-h.safetensors
✅ Downloaded successfully (808.3 MB)

⬇️ Downloading: ip-adapter-plus-face_sdxl_vit-h.safetensors
✅ Downloaded successfully (808.3 MB)

📂 CATEGORY: UPSCALE_MODELS
----------------------------------------

⬇️ Downloading: 4x_NMKD-Superscale-SP_178000_G.pth
✅ Downloaded successfully (63.9 MB)

⬇️ Downloading: OmniSR_X2_DIV2K.safetensors
✅ Downloaded successfully (1.6 MB)

⬇️ Downloading: 4x_foolhardy_Remacri.pth
✅ Downloaded successfully (63.9 MB)

📂 CATEGORY: INPAINT
----------------------------------------

⬇️ Downloading: MAT_Places512_G_fp16.safetensors
✅ Downloaded successfully (119.5 MB)

🎉 BATCH COMPLETE: 63 downloaded | 0 skipped | 0 failed


In [ ]:
# @title ⚡ Cell 2: Setup & Launch ComfyUI (Cloudflare Tunnel)
import os
import sys
import time
import re
import subprocess
import shutil
from pathlib import Path

# ============================================================
# ✏️ USER SETTINGS (Change these as needed)
# ============================================================
VRAM_FLAG = None           # "--lowvram" or "--highvram" or None for auto
SIMPLE_LOGS = True         # True = minimal logs, False = full logs
# ============================================================

os.chdir("/content")
BASE_DIR = Path("/content/ComfyUI")
MAIN_PY = BASE_DIR / "main.py"

print("="*60)
print("🚀 COMFYUI LAUNCHER")
print("="*60)

# ------------------------------------------------------------
# 1. Clone ComfyUI (if missing)
# ------------------------------------------------------------
if not MAIN_PY.exists():
    print("🧹 Core engine missing. Initializing git repo...")
    BASE_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(BASE_DIR)
    subprocess.run("git init -q", shell=True)
    subprocess.run("git remote add origin https://github.com/comfyanonymous/ComfyUI.git", shell=True)
    subprocess.run("git fetch --depth 1 origin master -q", shell=True)
    subprocess.run("git checkout -t origin/master -f -q", shell=True)
    print("✅ ComfyUI engine cloned.")
else:
    print("🛡️ Existing ComfyUI engine verified.")

# ------------------------------------------------------------
# 2. Install dependencies
# ------------------------------------------------------------
print("📦 Installing ComfyUI dependencies...")
os.chdir(BASE_DIR)
if (BASE_DIR / "requirements.txt").exists():
    subprocess.run("pip install -q -r requirements.txt", shell=True)
else:
    subprocess.run("pip install -q k-diffusion einops omegaconf torchvision", shell=True)

# ------------------------------------------------------------
# 3. Determine VRAM flag
# ------------------------------------------------------------
vram_flag = None
if VRAM_FLAG is not None:
    if VRAM_FLAG in ("--lowvram", "--highvram", "--novram", "--cpu"):
        vram_flag = VRAM_FLAG
        print(f"🔧 Using manual VRAM mode: {vram_flag}")
    else:
        print(f"⚠️ Ignoring invalid manual flag: '{VRAM_FLAG}'. Using auto.")
        VRAM_FLAG = None

if vram_flag is None:
    def get_gpu_memory_gb():
        try:
            result = subprocess.run("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits",
                                   shell=True, capture_output=True, text=True)
            if result.returncode == 0:
                total_mb = int(result.stdout.strip().split('\n')[0])
                return total_mb / 1024
        except:
            pass
        return None

    vram_gb = get_gpu_memory_gb()
    if vram_gb is None:
        print("⚠️ Could not detect GPU VRAM; using normal VRAM (no flag).")
    elif vram_gb < 6:
        vram_flag = "--lowvram"
        print(f"🔹 Detected {vram_gb:.1f} GB VRAM → using --lowvram")
    elif vram_gb > 16:
        vram_flag = "--highvram"
        print(f"🔹 Detected {vram_gb:.1f} GB VRAM → using --highvram")
    else:
        print(f"🔹 Detected {vram_gb:.1f} GB VRAM → using normal VRAM (no flag).")

# ------------------------------------------------------------
# 4. Clean up old processes
# ------------------------------------------------------------
print("🧹 Cleaning old processes...")
subprocess.run("pkill -9 -f cloudflared || true", shell=True)
subprocess.run("pkill -9 -f main.py || true", shell=True)
time.sleep(1)

# ------------------------------------------------------------
# 5. Install Cloudflare
# ------------------------------------------------------------
CF_BIN = "/usr/local/bin/cloudflared"
if not Path(CF_BIN).exists():
    print("🌐 Installing Cloudflare client...")
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared", shell=True)
    subprocess.run(f"chmod +x {CF_BIN}", shell=True)

# ------------------------------------------------------------
# 6. Start ComfyUI (in background) BEFORE tunnel
# ------------------------------------------------------------
print("🚀 Launching ComfyUI...")
cmd = ["python3", "main.py", "--listen", "0.0.0.0", "--port", "8188", "--enable-cors-header", "*"]
if vram_flag:
    cmd.append(vram_flag)

# We'll start ComfyUI and capture its output for filtering later
# but we need to run it in a way that we can still check its readiness.
# We'll use Popen with pipes.
comfy_process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

# Wait for ComfyUI to be ready (max 60 seconds)
print("⏳ Waiting for ComfyUI to start...")
ready = False
for i in range(60):
    time.sleep(1)
    # Check if process is still alive
    if comfy_process.poll() is not None:
        print("❌ ComfyUI process died early. Check logs above.")
        break
    # Check if port 8188 responds
    try:
        import socket
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(1)
        result = sock.connect_ex(('127.0.0.1', 8188))
        if result == 0:
            ready = True
            break
    except:
        pass
    print(".", end="", flush=True)
print()

if not ready:
    print("❌ ComfyUI failed to start within 60 seconds.")
    # Print the last few lines of output for debugging
    print("\n--- Last 20 lines of ComfyUI output ---")
    for i, line in enumerate(comfy_process.stdout):
        print(line, end="")
        if i > 20:
            break
    sys.exit(1)

print("✅ ComfyUI is ready on port 8188.")

# ------------------------------------------------------------
# 7. Start Cloudflare tunnel
# ------------------------------------------------------------
CF_LOG = "/content/cf.log"
if Path(CF_LOG).exists():
    Path(CF_LOG).unlink()

print("🌐 Starting Cloudflare tunnel...")
subprocess.Popen(f"{CF_BIN} tunnel --url http://127.0.0.1:8188 > {CF_LOG} 2>&1", shell=True)

public_url = None
print("⏳ Waiting for Cloudflare URL...", end="")
for _ in range(60):
    time.sleep(0.5)
    if Path(CF_LOG).exists():
        with open(CF_LOG, "r") as f:
            content = f.read()
            match = re.search(r"https://[-a-zA-Z0-9@:%._\+~#=]{1,256}\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break
    print(".", end="", flush=True)
print()

if public_url:
    # Verify tunnel is actually working by checking if the process is still alive
    tunnel_pid = subprocess.run("pgrep -f cloudflared", shell=True, capture_output=True, text=True)
    if tunnel_pid.returncode != 0 or not tunnel_pid.stdout.strip():
        print("⚠️ Cloudflare tunnel process died. Retrying once...")
        # Restart tunnel
        subprocess.run("pkill -9 -f cloudflared || true", shell=True)
        time.sleep(1)
        if Path(CF_LOG).exists():
            Path(CF_LOG).unlink()
        subprocess.Popen(f"{CF_BIN} tunnel --url http://127.0.0.1:8188 > {CF_LOG} 2>&1", shell=True)
        # Wait again
        public_url = None
        for _ in range(30):
            time.sleep(0.5)
            if Path(CF_LOG).exists():
                with open(CF_LOG, "r") as f:
                    content = f.read()
                    match = re.search(r"https://[-a-zA-Z0-9@:%._\+~#=]{1,256}\.trycloudflare\.com", content)
                    if match:
                        public_url = match.group(0)
                        break
        if public_url:
            print("✅ Tunnel restarted successfully.")
        else:
            print("❌ Tunnel restart failed.")
    else:
        print("✅ Tunnel process is alive.")

    if public_url:
        print(f"\n✅ COMFYUI PUBLIC URL: {public_url}")
        print("="*60)
        print("🔗 Share this URL to access ComfyUI from anywhere.")
        print("💡 If the link doesn't work, check:")
        print("   1. That ComfyUI is still running (see logs above).")
        print("   2. That your firewall/network allows Cloudflare tunnels.")
        print("="*60 + "\n")
    else:
        print("\n❌ Tunnel failed even after retry. Check the log:")
        if Path(CF_LOG).exists():
            with open(CF_LOG, "r") as f:
                print(f.read()[-500:])
        print("\n⚠️ You can still access ComfyUI locally at http://127.0.0.1:8188\n")
else:
    print("\n❌ Failed to obtain Cloudflare URL. Check the log:")
    if Path(CF_LOG).exists():
        with open(CF_LOG, "r") as f:
            print(f.read()[-500:])
    print("\n⚠️ You can still access ComfyUI locally at http://127.0.0.1:8188\n")

# ------------------------------------------------------------
# 8. Now stream the filtered logs from ComfyUI
# ------------------------------------------------------------
os.chdir(BASE_DIR)
os.environ["PYTHONWARNINGS"] = "ignore"

# Filter patterns
always_filter = re.compile(
    r"UserWarning|FutureWarning|xformers|triton|bitsandbytes|torch\.cuda|alembic|comfy_kitchen|aimdo|OpenGL_accelerate|Database upgraded",
    re.IGNORECASE
)

print("📡 Streaming ComfyUI logs (press Ctrl+C to stop)...")
if SIMPLE_LOGS:
    print("🔇 Simple logs mode: showing only errors and key info.\n")
else:
    print("🔊 Full logs mode: showing all output.\n")

def should_show_line(line):
    if always_filter.search(line):
        return False
    if SIMPLE_LOGS:
        important = re.search(r"(error|exception|traceback|listening on|starting comfyui|cloudflare|url|public link|tip|usage|failed|warning)", line, re.IGNORECASE)
        return bool(important)
    return True

try:
    for line in comfy_process.stdout:
        if should_show_line(line):
            print(line, end="")
except KeyboardInterrupt:
    print("\n🛑 ComfyUI stopped by user.")
finally:
    comfy_process.terminate()
    comfy_process.wait()
    subprocess.run("pkill -9 -f cloudflared || true", shell=True)
    print("🧹 Cleanup done.")

🚀 COMFYUI LAUNCHER
🧹 Core engine missing. Initializing git repo...
✅ ComfyUI engine cloned.
📦 Installing ComfyUI dependencies...
🔹 Detected 15.0 GB VRAM → using normal VRAM (no flag).
🧹 Cleaning old processes...
🌐 Installing Cloudflare client...
🚀 Launching ComfyUI...
⏳ Waiting for ComfyUI to start...
....................................................
✅ ComfyUI is ready on port 8188.
🌐 Starting Cloudflare tunnel...
⏳ Waiting for Cloudflare URL..............
✅ Tunnel process is alive.

✅ COMFYUI PUBLIC URL: https://exchanges-regarded-professional-offered.trycloudflare.com
🔗 Share this URL to access ComfyUI from anywhere.
💡 If the link doesn't work, check:
   1. That ComfyUI is still running (see logs above).
   2. That your firewall/network allows Cloudflare tunnels.

📡 Streaming ComfyUI logs (press Ctrl+C to stop)...
🔇 Simple logs mode: showing only errors and key info.

[WARNING] WARNING: You need pytorch with cu130 or higher to use optimized CUDA operations.
WARNING WARNING WARNING

In [ ]:
# @title
# Check if ComfyUI is running on port 8188
import subprocess
result = subprocess.run("curl -s -o /dev/null -w '%{http_code}' http://127.0.0.1:8188", shell=True, capture_output=True, text=True)
print(f"ComfyUI HTTP status: {result.stdout}")

# Check if cloudflared is still running
result = subprocess.run("pgrep -f cloudflared", shell=True, capture_output=True, text=True)
if result.returncode == 0:
    print("✅ cloudflared is running (PID: " + result.stdout.strip() + ")")
else:
    print("❌ cloudflared is NOT running")